# SLIS — Step 3: ML Model Training
Trains two models:
- **Risk Classifier** — predicts Low / Medium / High risk from IT1+IT2 signals only
- **Performance Predictor** — predicts final weighted score

Saves: `risk_classifier.joblib`, `performance_predictor.joblib`, `risk_thresholds.joblib`, `feature_columns.json`, `model_metrics.json`

> **Run notebook 01 first** to generate CSVs.

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold, KFold
from sklearn.metrics import classification_report, confusion_matrix, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
DATA_DIR = ROOT / 'data'
ML_DIR   = ROOT / 'ml'
ML_DIR.mkdir(exist_ok=True)
print('Paths:', DATA_DIR, ML_DIR)

## Load & Feature Engineering

In [ ]:
students   = pd.read_csv(DATA_DIR / 'students.csv')
attendance = pd.read_csv(DATA_DIR / 'attendance.csv')
scores     = pd.read_csv(DATA_DIR / 'scores.csv')
activity   = pd.read_csv(DATA_DIR / 'activity.csv')

att_agg = attendance.groupby('student_id').agg(
    avg_attendance = ('attendance_pct', 'mean'),
    att_month1     = ('attendance_pct', lambda x: x.iloc[0]),
    att_month2     = ('attendance_pct', lambda x: x.iloc[1]),
    att_month3     = ('attendance_pct', lambda x: x.iloc[2]),
    att_month4     = ('attendance_pct', lambda x: x.iloc[3]),
    att_trend      = ('attendance_pct', lambda x: float(np.polyfit([1,2,3,4], x.values, 1)[0])),
).reset_index()

sc_agg = scores.groupby('student_id').agg(
    avg_it1        = ('it1_score',      'mean'),
    avg_it2        = ('it2_score',      'mean'),
    avg_final      = ('final_score',    'mean'),
    avg_roll_w4    = ('roll_avg_w4',    'mean'),
    avg_roll_w8    = ('roll_avg_w8',    'mean'),
    avg_roll_w12   = ('roll_avg_w12',   'mean'),
    avg_assignment = ('assignment_avg', 'mean'),
    avg_weighted   = ('weighted_score', 'mean'),
    it1_std        = ('it1_score',      'std'),
    it2_std        = ('it2_score',      'std'),
    it1_range      = ('it1_score',      lambda x: x.max() - x.min()),
    it2_range      = ('it2_score',      lambda x: x.max() - x.min()),
).reset_index()

it1_means = scores.groupby('student_id')['it1_score'].mean()
it2_means = scores.groupby('student_id')['it2_score'].mean()
sc_agg['it1_it2_delta'] = (it2_means - it1_means).values

master = students.merge(att_agg, on='student_id').merge(sc_agg, on='student_id').merge(activity, on='student_id')
master['engagement_score'] = (master['lms_logins_per_week']*0.4 + master['forum_posts']*0.3 +
                               master['resources_accessed']*0.2 + (master['avg_session_minutes']/60)*0.1)
master['it1_it2_slope'] = (master['avg_it2'] - master['avg_it1']) / 4.0
master['early_score']   = (master['avg_it1'] + master['avg_it2']) / 2.0

p33 = master['early_score'].quantile(0.33)
p66 = master['early_score'].quantile(0.66)
master['risk_label'] = master['early_score'].apply(lambda x: 0 if x>=p66 else (1 if x>=p33 else 2))

print(f'Master shape: {master.shape}')
print(f'Risk p33={p33:.2f}  p66={p66:.2f}')
print('Risk dist:', master['risk_label'].value_counts().sort_index().to_dict())

## Model A — Risk Classifier

In [ ]:
clf_features = [
    'avg_attendance','att_month1','att_month2','att_trend','engagement_score','gpa_start',
    'avg_it1','avg_it2','it1_it2_delta','it1_it2_slope','avg_roll_w4','avg_roll_w8',
    'it1_std','it2_std','it1_range','it2_range','lms_logins_per_week','forum_posts',
]

X_clf = master[clf_features].values
y_clf = master['risk_label'].values
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

clf_candidates = {
    'Random Forest': Pipeline([('sc', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=3,
                                       class_weight='balanced', random_state=42, n_jobs=-1))]),
    'Gradient Boosting': Pipeline([('sc', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.08,
                                           subsample=0.8, random_state=42))]),
    'Logistic Regression': Pipeline([('sc', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced', random_state=42))]),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_clf_name, best_clf_score, best_clf_model = None, 0.0, None

for name, pipe in clf_candidates.items():
    cv = cross_val_score(pipe, X_clf, y_clf, cv=skf, scoring='f1_macro')
    print(f'  {name:25s}  CV macro-F1: {cv.mean():.4f} ± {cv.std():.4f}')
    if cv.mean() > best_clf_score:
        best_clf_score, best_clf_name, best_clf_model = cv.mean(), name, pipe

print(f'\nBest: {best_clf_name}  (CV macro-F1={best_clf_score:.4f})')

In [ ]:
best_clf_model.fit(X_tr_c, y_tr_c)
y_pred_c = best_clf_model.predict(X_te_c)
print(f'Test accuracy: {(y_pred_c==y_te_c).mean():.4f}')
print(classification_report(y_te_c, y_pred_c, target_names=['Low','Medium','High']))

y_proba_c = best_clf_model.predict_proba(X_te_c)
max_conf  = y_proba_c.max(axis=1)
print(f'Mean confidence: {max_conf.mean():.3f}')
print(f'Low conf (<60%): {(max_conf<0.60).sum()} | High conf (>85%): {(max_conf>0.85).sum()}')

In [ ]:
# Save model + predictions + thresholds
y_pred_all  = best_clf_model.predict(X_clf)
y_proba_all = best_clf_model.predict_proba(X_clf)
risk_names  = {0:'Low', 1:'Medium', 2:'High'}

clf_output = pd.DataFrame({
    'student_id':     master['student_id'].values,
    'true_risk':      [risk_names[r] for r in y_clf],
    'predicted_risk': [risk_names[r] for r in y_pred_all],
    'conf_low':       y_proba_all[:,0].round(3),
    'conf_medium':    y_proba_all[:,1].round(3),
    'conf_high':      y_proba_all[:,2].round(3),
    'max_confidence': y_proba_all.max(axis=1).round(3),
    'correct':        (y_pred_all == y_clf),
})
clf_output.to_csv(ML_DIR / 'risk_predictions_full.csv', index=False)
joblib.dump(best_clf_model, ML_DIR / 'risk_classifier.joblib')
joblib.dump({'p33': p33, 'p66': p66}, ML_DIR / 'risk_thresholds.joblib')
print('[✓] Saved: risk_classifier.joblib, risk_thresholds.joblib, risk_predictions_full.csv')

## Model B — Performance Predictor (Regression)

In [ ]:
reg_features = [
    'avg_attendance','att_month1','att_month2','att_trend','engagement_score','gpa_start',
    'avg_it1','avg_it2','it1_it2_delta','it1_it2_slope','avg_roll_w4','avg_roll_w8',
    'it1_std','it2_std','it1_range','it2_range','lms_logins_per_week','forum_posts',
    'resources_accessed','avg_session_minutes',
]

X_reg = master[reg_features].values
y_reg = master['avg_weighted'].values
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_candidates = {
    'Random Forest Regressor': Pipeline([('sc', StandardScaler()),
        ('reg', RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=3, random_state=42, n_jobs=-1))]),
    'Gradient Boosting Regressor': Pipeline([('sc', StandardScaler()),
        ('reg', GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.08, subsample=0.8, random_state=42))]),
    'Ridge Regression': Pipeline([('sc', StandardScaler()), ('reg', Ridge(alpha=0.5))]),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
best_reg_name, best_reg_rmse, best_reg_model = None, float('inf'), None

for name, pipe in reg_candidates.items():
    cv_neg  = cross_val_score(pipe, X_reg, y_reg, cv=kf, scoring='neg_mean_squared_error')
    cv_rmse = np.sqrt(-cv_neg).mean()
    print(f'  {name:32s}  CV RMSE: {cv_rmse:.4f}')
    if cv_rmse < best_reg_rmse:
        best_reg_rmse, best_reg_name, best_reg_model = cv_rmse, name, pipe

print(f'\nBest: {best_reg_name}  (CV RMSE={best_reg_rmse:.4f})')

In [ ]:
best_reg_model.fit(X_tr_r, y_tr_r)
y_pred_r  = best_reg_model.predict(X_te_r)
test_rmse = float(np.sqrt(mean_squared_error(y_te_r, y_pred_r)))
test_r2   = float(r2_score(y_te_r, y_pred_r))
errors    = y_pred_r - y_te_r

print(f'Test RMSE : {test_rmse:.4f}')
print(f'Test R²   : {test_r2:.4f}')
print(f'Mean error: {errors.mean():+.3f}  (bias)')
print(f'Within ±5 : {(np.abs(errors)<=5).mean():.1%}')
print(f'Within ±10: {(np.abs(errors)<=10).mean():.1%}')

joblib.dump(best_reg_model, ML_DIR / 'performance_predictor.joblib')
print('[✓] Saved: performance_predictor.joblib')

## Save Metadata

In [ ]:
feature_columns = {'risk_classifier': clf_features, 'performance_predictor': reg_features}
with open(ML_DIR / 'feature_columns.json', 'w') as f:
    json.dump(feature_columns, f, indent=2)

metrics = {
    'classifier': {
        'model_name':    best_clf_name,
        'cv_f1_macro':   round(best_clf_score, 4),
        'test_accuracy': round(float((y_pred_c==y_te_c).mean()), 4),
        'features':      clf_features,
        'classes':       ['Low','Medium','High'],
    },
    'regressor': {
        'model_name': best_reg_name,
        'cv_rmse':    round(best_reg_rmse, 4),
        'test_rmse':  round(test_rmse, 4),
        'test_r2':    round(test_r2, 4),
        'features':   reg_features,
    },
}
with open(ML_DIR / 'model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('[✓] Saved: feature_columns.json, model_metrics.json')
print('\nAll models saved to:', ML_DIR)
import os
for f in sorted(ML_DIR.glob('*.joblib')):
    print(f'  {f.name:45s} {f.stat().st_size/1024:.1f} KB')